# PickleSight: Pickleball Pose Analytics with YOLOv8-Pose

Detects and tracks players with `yolov8n-pose`, measures the primary player's arm span, stance width and 8 joint angles (shoulders, elbows, hips, knees), and renders `output_video.mp4` with a semi-transparent HUD, a colourful skeleton and joint-angle labels.

**Run order:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Upload the video when the upload cell asks for it.

## 1. Install dependencies

In [ ]:
# lap is required by the ByteTrack tracker; installing it up front avoids a mid-run auto-install
!pip install -q ultralytics opencv-python-headless lap pandas numpy

## 2. Upload the input video

In [ ]:
import os
from google.colab import files

# Optional: set to a video already in /content (e.g. after a runtime restart) to skip the upload
VIDEO_PATH = ""

if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    video_path = VIDEO_PATH
    print("Using existing video:", video_path)
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video was uploaded.")
    video_path = os.path.join("/content", next(iter(uploaded)))
    print("Uploaded video:", video_path)

## 3. Verify the video

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open the video: {video_path}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print("Video verified!")
print(f"Resolution: {width} x {height}")
print(f"FPS: {fps}")
print(f"Frames: {frames}")
print(f"Duration: {frames / fps:.2f} seconds")
print(f"Size: {os.path.getsize(video_path) / (1024**2):.2f} MB")

## 4. Check the environment

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

## 5. Pose estimation, tracking and keypoint extraction

In [ ]:
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from IPython.display import display

csv_path = "/content/picklesight_phase3_keypoints.csv"
video_out = "/content/picklesight_phase3_tracked.mp4"

# A fresh model each run so ByteTrack IDs start clean
model = YOLO("yolov8n-pose.pt")

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(video_out, fourcc, fps, (width, height))

keypoint_names = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

rows = []
frame_idx = 0

print(f"Processing {total_frames} frames...")

while True:
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        imgsz=960,
        conf=0.25,
        verbose=False
    )

    result = results[0]
    writer.write(result.plot())

    if result.keypoints is not None and result.boxes is not None:
        xy = result.keypoints.xy.cpu().numpy()
        confs = result.keypoints.conf

        if confs is not None:
            confs = confs.cpu().numpy()
        else:
            confs = np.ones(xy.shape[:2], dtype=float)

        if result.boxes.id is not None:
            track_ids = result.boxes.id.cpu().numpy().astype(int)

            for person_idx, track_id in enumerate(track_ids):
                for kp_idx, kp_name in enumerate(keypoint_names):
                    x, y = xy[person_idx, kp_idx]
                    rows.append({
                        "frame": frame_idx,
                        "track_id": int(track_id),
                        "keypoint": kp_name,
                        "x": float(x),
                        "y": float(y),
                        "confidence": float(confs[person_idx, kp_idx])
                    })

    frame_idx += 1

    if frame_idx % 25 == 0 or frame_idx == total_frames:
        print(f"Processed {frame_idx}/{total_frames} frames")

cap.release()
writer.release()

kp_df = pd.DataFrame(rows)
if kp_df.empty:
    raise RuntimeError("No tracked people were found. Check the video or lower the detection threshold.")
kp_df.to_csv(csv_path, index=False)

print("\nKeypoint extraction complete!")
print("Frames processed:", frame_idx)
print("Keypoint CSV:", csv_path)
print("CSV shape:", kp_df.shape)
print("Tracked video:", video_out)
print("Unique track IDs:", kp_df["track_id"].nunique())

display(kp_df.head())

## 6. Clean keypoints and calculate body metrics

In [ ]:
import numpy as np
import pandas as pd
import os

KEYPOINTS_CSV = "/content/picklesight_phase3_keypoints.csv"
METRICS_CSV = "/content/picklesight_metrics_v2.csv"

assert os.path.exists(KEYPOINTS_CSV), (
    f"Missing file: {KEYPOINTS_CSV}. "
    "Run the keypoint extraction cell first."
)

kp = pd.read_csv(KEYPOINTS_CSV)

required = {"frame", "track_id", "keypoint", "x", "y", "confidence"}
assert required.issubset(kp.columns), "The keypoint CSV is missing required columns."

# Remove unreliable coordinates before smoothing.
kp = kp.sort_values(["track_id", "keypoint", "frame"]).reset_index(drop=True)
low_confidence = kp["confidence"] < 0.50
kp.loc[low_confidence, ["x", "y"]] = np.nan

# Smooth each joint separately, restarting after long detection gaps.
group_cols = ["track_id", "keypoint"]
gap = kp.groupby(group_cols)["frame"].diff().fillna(1) > 5
kp["segment"] = gap.groupby([kp["track_id"], kp["keypoint"]]).cumsum()

for axis in ["x", "y"]:
    kp[f"{axis}_smooth"] = (
        kp.groupby(group_cols + ["segment"])[axis]
          .transform(lambda s: s.ewm(alpha=0.35, adjust=False).mean())
          .where(kp[axis].notna())
    )

# Convert keypoints to one row per frame and track.
wide = kp.pivot(
    index=["frame", "track_id"],
    columns="keypoint",
    values=["x_smooth", "y_smooth"]
)
wide.columns = [f"{joint}_{axis[0]}" for axis, joint in wide.columns]
wide = wide.reset_index()

joint_names = set(kp["keypoint"].unique())
needed = {
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
}
missing = needed - joint_names
assert not missing, f"Missing expected joints: {sorted(missing)}"


def point(joint):
    return wide[[f"{joint}_x", f"{joint}_y"]].to_numpy(dtype=float)


def distance(a, b):
    pa, pb = point(a), point(b)
    d = np.linalg.norm(pa - pb, axis=1)
    return np.where(np.isfinite(pa).all(axis=1) &
                    np.isfinite(pb).all(axis=1), d, np.nan)


def joint_angle(a, b, c):
    pa, pb, pc = point(a), point(b), point(c)
    ba, bc = pa - pb, pc - pb
    denom = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1)
    numerator = np.einsum("ij,ij->i", ba, bc)
    cosine = numerator / np.where(denom > 1e-6, denom, np.nan)
    valid = (np.isfinite(pa).all(axis=1) &
             np.isfinite(pb).all(axis=1) &
             np.isfinite(pc).all(axis=1) & (denom > 1e-6))
    return np.where(valid, np.degrees(np.arccos(np.clip(cosine, -1, 1))), np.nan)


metrics = wide[["frame", "track_id"]].copy()

for side in ["left", "right"]:
    metrics[f"{side}_shoulder_angle"] = joint_angle(
        f"{side}_hip", f"{side}_shoulder", f"{side}_elbow"
    )
    metrics[f"{side}_elbow_angle"] = joint_angle(
        f"{side}_shoulder", f"{side}_elbow", f"{side}_wrist"
    )
    metrics[f"{side}_hip_angle"] = joint_angle(
        f"{side}_shoulder", f"{side}_hip", f"{side}_knee"
    )
    metrics[f"{side}_knee_angle"] = joint_angle(
        f"{side}_hip", f"{side}_knee", f"{side}_ankle"
    )

metrics["arm_span_px"] = distance("left_wrist", "right_wrist")
metrics["stance_width_px"] = distance("left_ankle", "right_ankle")
metrics["shoulder_width_px"] = distance("left_shoulder", "right_shoulder")

# Torso length (shoulder to hip) stays stable when the player turns sideways
metrics["torso_px"] = pd.DataFrame({
    "left": distance("left_shoulder", "left_hip"),
    "right": distance("right_shoulder", "right_hip")
}).mean(axis=1, skipna=True)

valid_torso = metrics["torso_px"] > 1
metrics["arm_span_rel_torso"] = (
    metrics["arm_span_px"] / metrics["torso_px"].where(valid_torso)
)
metrics["stance_width_rel_torso"] = (
    metrics["stance_width_px"] / metrics["torso_px"].where(valid_torso)
)

# Primary player: largest median torso among tracks visible in at least half the frames.
# Set PRIMARY_TRACK_ID to an integer to force a specific player.
PRIMARY_TRACK_ID = None

frame_count = metrics["frame"].nunique()
summary = metrics.groupby("track_id").agg(
    visible_frames=("frame", "nunique"),
    median_torso_px=("torso_px", "median")
)
summary["coverage"] = summary["visible_frames"] / frame_count
candidates = summary[summary["coverage"] >= 0.50].dropna(
    subset=["median_torso_px"]
)

if PRIMARY_TRACK_ID is not None:
    assert PRIMARY_TRACK_ID in summary.index, f"Track {PRIMARY_TRACK_ID} was not found."
    primary_id = int(PRIMARY_TRACK_ID)
elif candidates.empty:
    raise ValueError("No track has enough valid torso measurements.")
else:
    primary_id = int(candidates["median_torso_px"].idxmax())

metrics["is_primary"] = metrics["track_id"] == primary_id
metrics.to_csv(METRICS_CSV, index=False)

print("Metrics calculated successfully.")
print("Low-confidence keypoints masked:", int(low_confidence.sum()))
print("Primary track selected:", primary_id)
print("Metrics saved to:", METRICS_CSV)
print("Number of joint-angle columns:",
      sum(c.endswith("_angle") for c in metrics.columns))
display(summary.sort_values("visible_frames", ascending=False).head(8).round(2))

## 7. Render the annotated output video

In [ ]:
import cv2
import pandas as pd
import numpy as np
import os
import time
import subprocess

# --------------------------------------------------
# 1. FILE PATHS AND SETTINGS
# --------------------------------------------------
VIDEO = video_path
KEYPOINTS_CSV = "/content/picklesight_phase3_keypoints.csv"
METRICS_CSV = "/content/picklesight_metrics_v2.csv"

TEMP_VIDEO = "/content/output_video_temp.mp4"
OUTPUT_VIDEO = "/content/output_video.mp4"

for path in [VIDEO, KEYPOINTS_CSV, METRICS_CSV]:
    assert os.path.exists(path), f"File not found: {path}"

kp = pd.read_csv(KEYPOINTS_CSV)
metrics = pd.read_csv(METRICS_CSV)

required_kp = {
    "frame", "track_id", "keypoint", "x", "y", "confidence"
}
required_metrics = {
    "frame", "track_id", "is_primary",
    "left_shoulder_angle", "left_elbow_angle",
    "left_hip_angle", "left_knee_angle",
    "right_shoulder_angle", "right_elbow_angle",
    "right_hip_angle", "right_knee_angle",
    "arm_span_px", "stance_width_px", "torso_px"
}

assert required_kp.issubset(kp.columns), (
    f"Missing keypoint CSV columns: {required_kp - set(kp.columns)}"
)
assert required_metrics.issubset(metrics.columns), (
    f"Missing metrics CSV columns: {required_metrics - set(metrics.columns)}"
)

# Primary player chosen in the metrics cell
PRIMARY_ID = int(metrics.loc[metrics["is_primary"], "track_id"].iloc[0])
print("Primary player: ID", PRIMARY_ID)

# Use metrics for the selected player.
primary_metrics = metrics[metrics["track_id"] == PRIMARY_ID]
metric_lookup = primary_metrics.set_index("frame").to_dict("index")

# Group keypoints by frame for efficient rendering.
frame_keypoints = {
    int(frame): group
    for frame, group in kp.groupby("frame")
}

# --------------------------------------------------
# 2. OPEN INPUT VIDEO AND CREATE WRITER
# --------------------------------------------------
cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f"Could not open input video: {VIDEO}"

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    TEMP_VIDEO, fourcc, video_fps, (width, height)
)
assert writer.isOpened(), "Could not create temporary video."

# --------------------------------------------------
# 3. SKELETON SETTINGS
# --------------------------------------------------
connections = [
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle")
]

colors = [
    (255, 100, 50),
    (50, 220, 255),
    (100, 255, 100),
    (255, 80, 200),
    (50, 150, 255),
    (200, 200, 50)
]

angle_specs = [
    ("left_shoulder_angle", "left_shoulder", "L Shoulder"),
    ("left_elbow_angle", "left_elbow", "L Elbow"),
    ("left_hip_angle", "left_hip", "L Hip"),
    ("left_knee_angle", "left_knee", "L Knee"),
    ("right_shoulder_angle", "right_shoulder", "R Shoulder"),
    ("right_elbow_angle", "right_elbow", "R Elbow"),
    ("right_hip_angle", "right_hip", "R Hip"),
    ("right_knee_angle", "right_knee", "R Knee")
]

LABEL_CODES = {
    "left_shoulder": "LS",
    "left_elbow": "LE",
    "left_hip": "LH",
    "left_knee": "LK",
    "right_shoulder": "RS",
    "right_elbow": "RE",
    "right_hip": "RH",
    "right_knee": "RK"
}

FONT = cv2.FONT_HERSHEY_SIMPLEX


def draw_text(
    img, text, position,
    scale=0.65, color=(255, 255, 255), thickness=2
):
    cv2.putText(
        img, str(text), position,
        FONT, scale, color, thickness, cv2.LINE_AA
    )


def draw_joint_label(img, value, joint, coords, placed, scale=0.55):
    """Draw a joint-angle label with a dark background and avoid
    previously placed label boxes where possible.
    """
    xj, yj = coords[joint]
    label = f"{LABEL_CODES[joint]} {value:.0f}"

    thickness = 2
    (tw, th), baseline = cv2.getTextSize(
        label, FONT, scale, thickness
    )

    gap = 5
    box_w = tw + 2 * gap
    box_h = th + baseline + 2 * gap

    # Place labels outside the body: joints left of the body centre get
    # labels on the left, joints right of it get labels on the right.
    # Screen position is used (not anatomical side) so labels stay outside
    # the body whether the player faces towards or away from the camera.
    body_cx = np.mean([p[0] for p in coords.values()])
    if xj < body_cx:
        preferred_x = xj - box_w - 12
    else:
        preferred_x = xj + 12

    # Clamp horizontally to the image.
    box_x = int(np.clip(
        preferred_x, 2, max(2, img.shape[1] - box_w - 2)
    ))

    # Start near the joint and search nearby vertical positions.
    max_y = max(2, img.shape[0] - box_h - 2)
    start_y = int(np.clip(yj - box_h // 2, 2, max_y))

    offsets = [0]
    for step in range(1, 25):
        offsets.extend([step * (box_h + 3),
                        -step * (box_h + 3)])

    chosen_box = None

    for offset in offsets:
        box_y = start_y + offset
        if box_y < 2 or box_y > max_y:
            continue

        box = (
            box_x,
            box_y,
            box_x + box_w,
            box_y + box_h
        )

        overlaps = any(
            box[0] < other[2] and box[2] > other[0]
            and box[1] < other[3] and box[3] > other[1]
            for other in placed
        )

        if not overlaps:
            chosen_box = box
            break

    # If every position is occupied, fall back to the position next to the joint.
    if chosen_box is None:
        box_y = start_y
        chosen_box = (
            box_x, box_y,
            box_x + box_w, box_y + box_h
        )

    placed.append(chosen_box)

    x1, y1, x2, y2 = chosen_box
    text_x = x1 + gap
    text_y = y1 + gap + th

    # Draw a connector from the joint to the label box.
    anchor_x = x1 if xj < x1 else x2
    anchor_y = int(np.clip(yj, y1, y2))

    cv2.line(
        img, (xj, yj), (anchor_x, anchor_y),
        (0, 220, 255), 1, cv2.LINE_AA
    )

    # Dark background makes the text easier to read.
    cv2.rectangle(img, (x1, y1), (x2, y2), (20, 20, 20), -1)

    cv2.putText(
        img, label, (text_x, text_y),
        FONT, scale, (0, 255, 255),
        thickness, cv2.LINE_AA
    )


# --------------------------------------------------
# 4. RENDER VIDEO OVERLAY
# --------------------------------------------------
start_time = time.time()
frame_idx = 0

print(f"Rendering {total_frames} frames...")
print(f"Resolution: {width} x {height}")
print(f"Video FPS: {video_fps}")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        elapsed = frame_idx / video_fps
        render_fps = (
            (frame_idx + 1) / max(time.time() - start_time, 1e-6)
        )

        current_kp = frame_keypoints.get(frame_idx)
        coords = {}

        if current_kp is not None:
            current_kp = current_kp[
                (current_kp["track_id"] == PRIMARY_ID)
                & (current_kp["confidence"] >= 0.50)
            ]

            for _, row in current_kp.iterrows():
                name = row["keypoint"]
                x, y = int(row["x"]), int(row["y"])
                coords[name] = (x, y)

            # Draw skeleton connections.
            for i, (a, b) in enumerate(connections):
                if a in coords and b in coords:
                    cv2.line(
                        frame, coords[a], coords[b],
                        colors[i % len(colors)], 4, cv2.LINE_AA
                    )

            # Draw joints.
            for name, (x, y) in coords.items():
                color_index = sum(
                    ord(char) for char in name
                ) % len(colors)

                cv2.circle(
                    frame, (x, y), 6,
                    (255, 255, 255), -1, cv2.LINE_AA
                )
                cv2.circle(
                    frame, (x, y), 4,
                    colors[color_index], -1, cv2.LINE_AA
                )

        m = metric_lookup.get(frame_idx, {})

        # Semi-transparent HUD panel.
        panel_w = min(570, width - 30)
        panel_h = min(470, height - 30)

        overlay = frame.copy()
        cv2.rectangle(
            overlay, (15, 15),
            (15 + panel_w, 15 + panel_h),
            (20, 25, 35), -1
        )
        frame = cv2.addWeighted(
            overlay, 0.78, frame, 0.22, 0
        )

        draw_text(
            frame, "PICKLESIGHT | PLAYER PERFORMANCE",
            (32, 50), 0.72, (100, 230, 255), 2
        )
        draw_text(
            frame, f"Primary player: ID {PRIMARY_ID}", (32, 82)
        )
        draw_text(
            frame,
            f"Time: {elapsed:05.2f} / {total_frames/video_fps:.2f} s",
            (32, 112)
        )
        draw_text(
            frame, f"Render FPS: {render_fps:.1f}", (32, 142)
        )

        detected_count = (
            int(frame_keypoints[frame_idx]["track_id"].nunique())
            if frame_idx in frame_keypoints else 0
        )

        draw_text(
            frame, f"Tracked persons: {detected_count}",
            (300, 142), 0.60, (120, 255, 150)
        )

        draw_text(
            frame, "JOINT ANGLES",
            (32, 180), 0.66, (100, 230, 255)
        )

        y = 210
        placed = []

        for column, joint, label in angle_specs:
            value = m.get(column, np.nan)

            text = (
                f"{label}: {value:.1f} deg"
                if pd.notna(value)
                else f"{label}: N/A"
            )

            draw_text(frame, text, (32, y), 0.56)
            y += 27

            # Collision-aware labels beside visible joints.
            if pd.notna(value) and joint in coords:
                draw_joint_label(
                    frame, value, joint, coords, placed
                )

        arm = m.get("arm_span_px", np.nan)
        stance = m.get("stance_width_px", np.nan)

        draw_text(
            frame, "BODY MEASUREMENTS",
            (32, 435), 0.60, (100, 230, 255)
        )

        draw_text(
            frame,
            f"Arm span: {arm:.1f}px"
            if pd.notna(arm) else "Arm span: N/A",
            (32, 462), 0.55
        )

        draw_text(
            frame,
            f"Stance: {stance:.1f}px"
            if pd.notna(stance) else "Stance: N/A",
            (280, 462), 0.55
        )

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 25 == 0 or frame_idx == total_frames:
            print(f"Rendered {frame_idx}/{total_frames} frames")

finally:
    cap.release()
    writer.release()

# --------------------------------------------------
# 5. CHECK TEMPORARY VIDEO
# --------------------------------------------------
assert frame_idx > 0, "No frames were rendered."
assert os.path.exists(TEMP_VIDEO), "Temporary video was not created."
assert os.path.getsize(TEMP_VIDEO) > 0, "Temporary video is empty."

print("\nOverlay rendering complete.")
print("Frames rendered:", frame_idx)
print(
    "Temporary size (MB):",
    round(os.path.getsize(TEMP_VIDEO) / 1048576, 2)
)

# --------------------------------------------------
# 6. CONVERT TO H.264
# --------------------------------------------------
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", TEMP_VIDEO,
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "fast",
        "-crf", "20",
        "-movflags", "+faststart",
        "-an",
        OUTPUT_VIDEO
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-2500:])
    raise RuntimeError("H.264 conversion failed.")

assert os.path.exists(OUTPUT_VIDEO), "Final video was not created."
assert os.path.getsize(OUTPUT_VIDEO) > 0, "Final video is empty."
os.remove(TEMP_VIDEO)

print("\nH.264 conversion successful!")
print("Final output:", OUTPUT_VIDEO)
print(
    "Final size (MB):",
    round(os.path.getsize(OUTPUT_VIDEO) / 1048576, 2)
)

## 8. Inspect sample frames

In [ ]:
import cv2
import numpy as np
from IPython.display import display, Image

output_path = "/content/output_video.mp4"
cap = cv2.VideoCapture(output_path)

if not cap.isOpened():
    raise RuntimeError("Could not open the output video.")

frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_out = cap.get(cv2.CAP_PROP_FPS) or 30

# Four evenly spaced frames across the whole video
for frame_number in np.linspace(0, frame_total - 1, 4).astype(int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
    success, frame = cap.read()

    if not success:
        print(f"Could not read frame {frame_number}")
        continue

    preview = cv2.resize(frame, (1280, int(1280 * frame.shape[0] / frame.shape[1])))
    success, encoded = cv2.imencode(".jpg", preview)

    if success:
        print(f"Frame {frame_number} ({frame_number / fps_out:.2f} seconds)")
        display(Image(data=encoded.tobytes()))

cap.release()

## 9. Verify the output against the input video

In [ ]:
import cv2
import os

output_path = "/content/output_video.mp4"

if not os.path.exists(output_path):
    raise FileNotFoundError(f"Video not found: {output_path}")


def video_info(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open: {path}")
    info = (
        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        cap.get(cv2.CAP_PROP_FPS),
        int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    )
    cap.release()
    return info


in_w, in_h, in_fps, in_frames = video_info(video_path)
out_w, out_h, out_fps, out_frames = video_info(output_path)

print("Final video verified!")
print(f"Resolution: {out_w} x {out_h}")
print(f"FPS: {out_fps}")
print(f"Total frames: {out_frames}")
print(f"Duration: {out_frames / out_fps:.2f} seconds")
print(f"File size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB")

assert (out_w, out_h) == (in_w, in_h), "Output resolution differs from the input"
assert out_frames == in_frames, "Output frame count differs from the input"
assert abs(out_fps - in_fps) < 0.1, "Output frame rate differs from the input"

print("All technical checks passed!")

## 10. Download the results

In [ ]:
from google.colab import files

files.download("/content/output_video.mp4")
files.download("/content/picklesight_metrics_v2.csv")